# Chronic Urticaria – Labs + Symptom Notes Multi‑Task Model (Clinical‑Safe)
### Targets
- **Urticaria type** (3‑class)
- **Secondary disease risk** (thyroid + autoimmune; multi‑label)
- **Side‑effect / hypersensitivity risk** (**proxy** 3‑class)
- **Severity + uncertainty** (heteroscedastic regression on itching score)

### Core novelties implemented
1) **Task‑conditioned gated fusion** (separate gate per task) combining tabular (labs + structured symptoms) and text (symptom notes)
2) **Clinical consistency regularizer** (soft rule penalty)
3) **Uncertainty‑aware severity** (μ + σ²)
4) **Leakage guard**: we exclude provocation/test columns from inputs to avoid unrealistic accuracy

> Run in **Google Colab**. This is a research prototype for decision support.

## 0) Install & Imports

In [1]:

!pip -q install transformers accelerate scikit-learn pandas numpy tqdm joblib

import os, json, math, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
import joblib


In [2]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## 1) Load CSV
Choose **one** option.

In [3]:

# # Option A (Upload)
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = list(uploaded.keys())[0]
# print("Using:", CSV_PATH)


In [4]:

# Option B (Google Drive) - uncomment if needed
from google.colab import drive
drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/AURA/final_3000_dataset/final_3000_Risk_Dataset.csv"  # <-- change this
print("Using:", CSV_PATH)


Mounted at /content/drive
Using: /content/drive/MyDrive/AURA/final_3000_dataset/final_3000_Risk_Dataset.csv


In [5]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head(2)


Shape: (3000, 39)


,split,symptoms_raw,investigations_raw,CRP,FT4,IgE,VitD,Age,History of Chronic Urticaria,Sex,...,Which applies to your wheals/angioedema or both?,Which of the following applies to your symptoms of urticaria?,Which time of the day do the symptoms occur?,Symptoms of Autoinflamation:,Specify other allergy,Alpha Gal,Family History of Urticaria,Family history of thyroid diseases,Family history of autoimmune diseases,urticaria_type
0,train,"Recurrent urticaria, itching","CBC, CRP, FT4, IgE, Vitamin D",8.73,1.48,114.41,25.75,59,No,Female,...,Can induce through specific stimuli,Itching,No Significance,Recurrent unexplained Fever,Hair dye,Positive,Yes,Yes,Yes,Inducible
1,validation,"Itching, Wheals, Swelling","CBC, CRP, FT4, IgE, Vitamin D",10.67,1.40,204.41,38.58,43,No,Female,...,Can induce through specific stimuli,Itching,No Significance,Recurrent unexplained Fever,Prawn like food allergy,Not tested,No,No,No,Inducible


## 2) Define columns (Clinical‑safe inputs)
### Leakage guard (excluded)
We **exclude** provocation/test result fields because they encode post‑diagnosis information and can inflate accuracy.

In [6]:

# Text (notes)
TEXT_COLS = ["symptoms_raw", "investigations_raw"]

# Numeric labs/features (IMPORTANT: do NOT include itching score here; it's the severity target)
LAB_COLS = ["CRP","FT4","IgE","VitD","Age","Weight","Height","Diagnosed at the age of"]

# Family history (categorical)
# NOW includes thyroid/autoimmune family history as INPUT FEATURES.
# These are valid clinical priors — they become inputs, while the risk TARGETS
# are multi-signal composite proxies (see Cell 3), so there is no leakage.
FAM_COLS = [
    "Family History of Urticaria",
    "Family history of thyroid diseases",
    "Family history of autoimmune diseases",
]

# Structured symptom questionnaire features (categorical) – strong clinically relevant signal for urticaria type
SYM_COLS = [
 "Sex","History of Chronic Urticaria","Symptoms Of Urticaria","Duration of Symptoms of urticaria",
 "If Wheals are present","The shape of an individual wheal","Size of a single Wheal","No. of wheals",
 "Duration of wheal","Location","If  angioedema is present","Duration of angioedema",
 "Discomfort of Swelling","Affect of Swelling on Daily activities","Angioedema affect on appearance",
 "Overall affect of Swelling","Which applies to your wheals/angioedema or both?",
 "Which of the following applies to your symptoms of urticaria?",
 "Which time of the day do the symptoms occur?","Symptoms of Autoinflamation:",
 "Alpha Gal","Specify other allergy",
 # Drug-induced angioedema remission: relevant to side-effect/drug-reaction risk
 "Remission of Angioedema after discontinuation of the drug:",
]

# Columns to EXCLUDE (leakage/post-test)
LEAK_COLS = [
    "Type of Provocation",
    "Provocation Test results",
    "If wheals/angioedema or both can be induced through specific stimulation, which stimuli can be applied?"
]

TARGET_TYPE = "urticaria_type"
assert "split" in df.columns, "Expected a 'split' column with train/val/test."



# ---- Column existence guard (prevents KeyError if any column name differs) ----
def keep_existing(cols):
    return [c for c in cols if c in df.columns]

missing = {
    "TEXT_COLS": [c for c in TEXT_COLS if c not in df.columns],
    "LAB_COLS":  [c for c in LAB_COLS if c not in df.columns],
    "FAM_COLS":  [c for c in FAM_COLS if c not in df.columns],
    "SYM_COLS":  [c for c in SYM_COLS if c not in df.columns],
}
if any(len(v) for v in missing.values()):
    print("⚠️ Missing columns detected (they will be skipped):")
    for k,v in missing.items():
        if v: print(k, "->", v)

TEXT_COLS = keep_existing(TEXT_COLS)
LAB_COLS  = keep_existing(LAB_COLS)
FAM_COLS  = keep_existing(FAM_COLS)
SYM_COLS  = keep_existing(SYM_COLS)


## 3) Targets
### 3.1 Urticaria type
### 3.2 Secondary disease risk (multi‑label)
### 3.3 Side‑effect/hypersensitivity risk (proxy 3‑class)
### 3.4 Severity target (itching score) + mask

In [7]:

# --- Urticaria type mapping ---
type_classes = sorted(df[TARGET_TYPE].dropna().unique().tolist())
type2id = {c:i for i,c in enumerate(type_classes)}
id2type = {i:c for c,i in type2id.items()}
print("Urticaria types:", type_classes)
assert len(type_classes)==3, f"Expected 3 urticaria types, got {len(type_classes)}: {type_classes}"

def yn_to_int(x):
    if pd.isna(x): return 0
    s = str(x).strip().lower()
    return 1 if s in ["yes","y","true","1"] else 0

# --- Secondary disease risk (multi-signal clinical proxy targets) ---
# DESIGN RATIONALE:
#   Family history of thyroid/autoimmune diseases is now an INPUT FEATURE in FAM_COLS.
#   Risk TARGETS are composite proxy scores built from multiple independent signals.
#   This ensures the target has genuine predictive variation (AUC > 0.5) rather than
#   predicting family history from symptoms, which is random (AUC ≈ 0.5).
#
#   Thyroid risk proxy  (threshold ≥ 3 points):
#     2 pts: family history of thyroid disease
#     2 pts: FT4 outside reference range (bottom-10% or top-10%)
#     1 pt:  IgE > median
#     1 pt:  CRP > 75th percentile
#     1 pt:  VitD < 25th percentile (insufficiency)
#     1 pt:  angioedema present
#
#   Autoimmune risk proxy (threshold ≥ 3 points):
#     2 pts: family history of autoimmune disease
#     1 pt:  IgE > 75th percentile
#     1 pt:  angioedema present
#     1 pt:  Alpha Gal sensitisation
#     1 pt:  autoinflammation symptoms reported
#     1 pt:  chronic/long disease duration

_ige_med = df["IgE"].median()
_ige_q75 = df["IgE"].quantile(0.75)
_crp_q75 = df["CRP"].quantile(0.75)
_ft4_lo  = df["FT4"].quantile(0.10)   # low FT4 → possible hypothyroidism
_ft4_hi  = df["FT4"].quantile(0.90)   # high FT4 → possible hyperthyroidism
_vitd_lo = df["VitD"].quantile(0.25)  # bottom quartile = vitamin D insufficiency
_ft4_mid = (_ft4_lo + _ft4_hi) / 2   # safe default for missing FT4

def _make_thyroid_risk(row):
    """Score ≥ 3 → elevated thyroid autoimmunity risk (positive label)."""
    s = 0
    if yn_to_int(row.get("Family history of thyroid diseases", "No")): s += 2
    try:
        ft4 = float(row.get("FT4", _ft4_mid))
        if ft4 < _ft4_lo or ft4 > _ft4_hi: s += 2  # outside reference range
    except: pass
    try:
        if float(row.get("IgE", 0)) > _ige_med: s += 1
    except: pass
    try:
        if float(row.get("CRP", 0)) > _crp_q75: s += 1
    except: pass
    try:
        if float(row.get("VitD", _vitd_lo + 1)) < _vitd_lo: s += 1
    except: pass
    if "yes" in str(row.get("If  angioedema is present", "")).lower(): s += 1
    return 1 if s >= 3 else 0

def _make_autoimmune_risk(row):
    """Score ≥ 3 → elevated autoimmune comorbidity risk (positive label)."""
    s = 0
    if yn_to_int(row.get("Family history of autoimmune diseases", "No")): s += 2
    try:
        if float(row.get("IgE", 0)) > _ige_q75: s += 1
    except: pass
    if "yes" in str(row.get("If  angioedema is present", "")).lower(): s += 1
    if "yes" in str(row.get("Alpha Gal", "")).lower(): s += 1
    autoinf = str(row.get("Symptoms of Autoinflamation:", "")).strip().lower()
    if autoinf not in {"", "no", "none", "nan", "n/a", "-"}: s += 1
    dur = str(row.get("Duration of Symptoms of urticaria", "")).lower()
    if any(k in dur for k in ["year", "month", "chronic", "persistent"]): s += 1
    return 1 if s >= 3 else 0

df["risk_thyroid"]    = df.apply(_make_thyroid_risk,    axis=1)
df["risk_autoimmune"] = df.apply(_make_autoimmune_risk, axis=1)
print("Thyroid risk distribution:    ", df["risk_thyroid"].value_counts().to_dict())
print("Autoimmune risk distribution: ", df["risk_autoimmune"].value_counts().to_dict())
RISK_LABELS = ["risk_thyroid", "risk_autoimmune"]

# --- Side-effect / hypersensitivity risk proxy (richer 7-signal scoring) ---
# Scored on 7 binary clinical signals (total 0-8 points):
#   ① IgE level tier    (0/1/2)  ② angioedema presence   ③ angioedema duration
#   ④ Alpha Gal allergy          ⑤ other documented allergy
#   ⑥ severe daily-life impact  ⑦ severe swelling discomfort
# All features derivable from questionnaire; no post-diagnosis leakage.
ige_q75 = df["IgE"].quantile(0.75)   # more selective than q67 → better class separation
ige_q92 = df["IgE"].quantile(0.92)

def make_side_effect_proxy(row, q75, q92):
    # ① IgE level (0 / 1 / 2 points)
    ig = row.get("IgE", np.nan)
    ig_score = 0
    if not pd.isna(ig):
        if   ig >= q92: ig_score = 2
        elif ig >= q75: ig_score = 1

    # ② Angioedema presence
    ang = str(row.get("If  angioedema is present", "")).lower()
    ang_score = 1 if "yes" in ang else 0

    # ③ Angioedema duration (persistent / days / weeks → elevated risk)
    ang_dur = str(row.get("Duration of angioedema", "")).lower()
    dur_kw = ["day", "week", "month", "chronic", "persistent", "recurrent", "hour"]
    ang_dur_score = 1 if any(k in ang_dur for k in dur_kw) else 0

    # ④ Alpha Gal sensitisation
    alpha_score = 1 if "yes" in str(row.get("Alpha Gal", "")).lower() else 0

    # ⑤ Other documented allergy (non-trivial entry)
    other = str(row.get("Specify other allergy", "")).strip().lower()
    other_score = 0 if other in {"", "no", "none", "nan", "n/a", "nil", "-"} else 1

    # ⑥ Severe impact on daily activities
    daily = str(row.get("Affect of Swelling on Daily activities", "")).lower()
    daily_kw = ["severe", "extreme", "unable", "always", "completely", "significantly"]
    daily_score = 1 if any(k in daily for k in daily_kw) else 0

    # ⑦ Severe swelling discomfort
    discomfort = str(row.get("Discomfort of Swelling", "")).lower()
    dis_kw = ["severe", "extreme", "unbearable", "intense", "very"]
    discomfort_score = 1 if any(k in discomfort for k in dis_kw) else 0

    total = ig_score + ang_score + ang_dur_score + alpha_score + other_score + daily_score + discomfort_score
    # 0–1 → LOW (0),  2–3 → MODERATE (1),  4+ → HIGH (2)
    if total >= 4: return 2
    if total >= 2: return 1
    return 0

df["sidefx_proxy"] = df.apply(lambda r: make_side_effect_proxy(r, ige_q75, ige_q92), axis=1)
print("Sidefx proxy distribution:\n", df["sidefx_proxy"].value_counts().sort_index())

# --- Severity target (itching score) ---
df["sev"] = pd.to_numeric(df["Itching score"], errors="coerce")
df["sev_mask"] = (~df["sev"].isna()).astype(int)
df["sev"] = df["sev"].fillna(df["sev"].median())


Urticaria types: ['Inducible', 'Mixed', 'Spontaneous']
Thyroid risk distribution:     {1: 1628, 0: 1372}
Autoimmune risk distribution:  {1: 2317, 0: 683}
Sidefx proxy distribution:
 sidefx_proxy
0    1274
1    1378
2     348
Name: count, dtype: int64


## 4) Build modeling dataframe (exclude leakage columns from inputs)
We keep **notes + labs + structured symptoms + family history**.

In [8]:
from sklearn.model_selection import train_test_split

# Keep only necessary columns for modeling
use_cols = ["split"] + TEXT_COLS + LAB_COLS + FAM_COLS + SYM_COLS + [TARGET_TYPE] + RISK_LABELS + ["sidefx_proxy","sev","sev_mask"]
data = df[use_cols].copy()

# Fill text columns
for c in TEXT_COLS:
    data[c] = data[c].fillna("")

# Combine text into one note
data["note_text"] = (data["symptoms_raw"].astype(str) + " [SEP] " + data["investigations_raw"].astype(str)).str.strip()

# Train/val/test
# NOTE: the dataset uses 'validation' (not 'val') as the split label — match both to be safe
train_df = data[data["split"].str.lower()=="train"].reset_index(drop=True)
val_df   = data[data["split"].str.lower().isin(["val","validation"])].reset_index(drop=True)
test_df  = data[data["split"].str.lower()=="test"].reset_index(drop=True)

# If val_df is empty, create a validation set from the training data
if val_df.empty:
    print("Warning: Validation DataFrame is empty. Splitting training data for validation (80/20 split).")
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=train_df[TARGET_TYPE])
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

print("Splits:", train_df.shape, val_df.shape, test_df.shape)


Splits: (2062, 44) (484, 44) (454, 44)


## 5) Tabular preprocessing (RobustScaler + OneHotEncoder)
This is the key change to improve urticaria type accuracy: **include structured symptoms** as categorical features.

In [9]:

# Decide numeric vs categorical columns
NUM_COLS = LAB_COLS
CAT_COLS = FAM_COLS + SYM_COLS

# Fill numeric missing values using training median (no leakage)
for c in NUM_COLS:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    _med = train_df[c].median()
    for d in [train_df, val_df, test_df]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(_med)

# Fill categorical missing values using training mode (no leakage, avoids dominant "Unknown" category)
_train_mode = {c: train_df[c].fillna("Unknown").astype(str).mode()[0] for c in CAT_COLS}
for d in [train_df, val_df, test_df]:
    for c in CAT_COLS:
        d[c] = d[c].fillna(_train_mode[c]).astype(str)

preprocess = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ]
)

train_tab = preprocess.fit_transform(train_df)
val_tab   = preprocess.transform(val_df)
test_tab  = preprocess.transform(test_df)

# Convert sparse -> dense float32 for PyTorch
train_tab = train_tab.toarray().astype(np.float32)
val_tab   = val_tab.toarray().astype(np.float32)
test_tab  = test_tab.toarray().astype(np.float32)

print("Tab dim:", train_tab.shape[1])


Tab dim: 154


## 6) Tokenizer & Dataset

In [10]:

from transformers import AutoTokenizer

# clinical encoder
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 384  # increased from 256 to capture more of the clinical notes


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [11]:

from torch.utils.data import WeightedRandomSampler

class CUData(Dataset):
    def __init__(self, df, tab_arr):
        self.df = df.reset_index(drop=True)
        self.tab = tab_arr.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["note_text"]

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        y_type = type2id[row[TARGET_TYPE]]
        y_risk = np.array([row["risk_thyroid"], row["risk_autoimmune"]], dtype=np.float32)
        y_side = int(row["sidefx_proxy"])

        sev = float(row["sev"])
        sev_mask = int(row["sev_mask"])

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "tab": torch.tensor(self.tab[idx]),
            "y_type": torch.tensor(y_type, dtype=torch.long),
            "y_risk": torch.tensor(y_risk, dtype=torch.float32),
            "y_side": torch.tensor(y_side, dtype=torch.long),
            "sev": torch.tensor(sev, dtype=torch.float32),
            "sev_mask": torch.tensor(sev_mask, dtype=torch.float32),
        }

train_ds = CUData(train_df, train_tab)
val_ds   = CUData(val_df, val_tab)
test_ds  = CUData(test_df, test_tab)

# ── Balanced batch sampling (WeightedRandomSampler) ─────────────────────────
# Addresses class imbalance in urticaria type (Inducible << Mixed ≈ Spontaneous).
# Each training batch is drawn so all 3 type classes appear at equal expected
# frequency — prevents the model from ignoring minority classes during training.
# This works on top of focal loss + class weights for maximum imbalance handling.
_type_ids     = train_df[TARGET_TYPE].map(type2id).values
_class_counts = np.bincount(_type_ids, minlength=len(type_classes)).astype(float)
_class_w      = 1.0 / (_class_counts + 1e-9)
_sample_w     = torch.tensor(_class_w[_type_ids], dtype=torch.float64)

_sampler = WeightedRandomSampler(
    weights=_sample_w,
    num_samples=len(train_ds),
    replacement=True,
)
print("Train class counts:", {id2type[i]: int(_class_counts[i]) for i in range(len(type_classes))})
print("Sample weights     :", {id2type[i]: round(float(_class_w[i]), 4) for i in range(len(type_classes))})

# Side-effect: also weight samples by HIGH/MODERATE/LOW to reduce HIGH under-representation
_side_ids    = train_df["sidefx_proxy"].values
_side_counts = np.bincount(_side_ids, minlength=3).astype(float)
_side_cw     = 1.0 / (_side_counts + 1e-9)
# Combined weight: type imbalance × side-effect imbalance (geometric blend)
_combined_w  = torch.tensor(
    np.sqrt(_class_w[_type_ids] * _side_cw[_side_ids]), dtype=torch.float64
)
_sampler_combined = WeightedRandomSampler(
    weights=_combined_w,
    num_samples=len(train_ds),
    replacement=True,
)
print("Side counts:", {["LOW","MOD","HIGH"][i]: int(_side_counts[i]) for i in range(3)})

train_loader = DataLoader(train_ds, batch_size=16, sampler=_sampler_combined)  # sampler replaces shuffle
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


Train class counts: {'Inducible': 371, 'Mixed': 841, 'Spontaneous': 850}
Sample weights     : {'Inducible': 0.0027, 'Mixed': 0.0012, 'Spontaneous': 0.0012}
Side counts: {'LOW': 861, 'MOD': 960, 'HIGH': 241}


## 7) Model – Task‑Conditioned Gated Fusion + Multi‑Head + Uncertainty

In [12]:

from transformers import AutoModel


class AttentionPooling(nn.Module):
    """Learned single-head attention pooling over BERT token sequence.

    Assigns each token a scalar relevance score; output is a weighted sum.
    More expressive than CLS-only or naive mean pooling: the network learns
    to focus on clinically informative tokens (symptom terms, lab findings).
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.Tanh(),
            nn.Linear(d_model // 4, 1, bias=False),
        )

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        scores = self.score(hidden_states).squeeze(-1)          # (B, T)
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)   # (B, T, 1)
        return (hidden_states * weights).sum(dim=1)             # (B, D)


class GatedFusionMTL(nn.Module):
    """Task-Conditioned Gated Fusion MTL model for Chronic Urticaria risk profiling.

    Core novelties
    ──────────────
    ① Per-task gating        — independent gate per head learns optimal text/lab
                               blend for each clinical task
    ② Heteroscedastic sev.   — outputs μ + log(σ²) for calibrated uncertainty
    ③ Clinical consistency   — soft rule penalty (high IgE ≠ LOW side-effect risk)
    ④ Uncertainty MTL loss   — learned log-variances auto-balance task losses
                               (Kendall et al., NeurIPS 2018)
    ⑤ Attention pooling      — tokens weighted by learned clinical relevance score
    ⑥ Residual tab encoder   — skip-connections for richer lab/symptom features
    """

    def __init__(self, tab_dim: int, text_model_name: str, hidden: int = 512, dropout: float = 0.3):
        super().__init__()
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        text_dim = self.text_encoder.config.hidden_size

        # ── Learned attention pooling over BERT token sequence ────────────────
        self.attn_pool = AttentionPooling(text_dim)

        # ── Residual tabular encoder (deep lab/symptom representation) ────────
        self.tab_proj = nn.Linear(tab_dim, hidden)
        self.tab_res1 = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout)
        )
        self.tab_res2 = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout)
        )

        # ── Text projection ───────────────────────────────────────────────────
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout)
        )

        # ── Shared bottom (common knowledge before task-specific gates) ────────
        self.shared = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout)
        )

        # ── Per-task gating networks ───────────────────────────────────────────
        def gate_net():
            return nn.Sequential(
                nn.Linear(hidden * 2, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden),     nn.Sigmoid(),
            )

        self.gate_type = gate_net()
        self.gate_risk = gate_net()
        self.gate_side = gate_net()
        self.gate_sev  = gate_net()

        # ── Post-fusion LayerNorm per task ─────────────────────────────────────
        self.norm_type = nn.LayerNorm(hidden)
        self.norm_risk = nn.LayerNorm(hidden)
        self.norm_side = nn.LayerNorm(hidden)
        self.norm_sev  = nn.LayerNorm(hidden)

        # ── Task heads: 2-layer MLP (richer decision boundary) ────────────────
        def mlp_head(n_out: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Linear(hidden, hidden // 2), nn.GELU(),
                nn.Linear(hidden // 2, n_out),
            )

        self.head_type    = mlp_head(3)   # urticaria type  (3-class)
        self.head_risk    = mlp_head(2)   # secondary risk  (multi-label)
        self.head_side    = mlp_head(3)   # side-effect     (LOW / MOD / HIGH)
        self.head_sev_mu  = nn.Linear(hidden, 1)   # severity mean
        self.head_sev_lv  = nn.Linear(hidden, 1)   # severity log-variance

        # ── Uncertainty task-weighting (Kendall et al. 2018) ──────────────────
        # log(σ²) per task [type, risk, side, sev]: learned to auto-balance losses
        self.log_var = nn.Parameter(torch.zeros(4))

    def _encode_tab(self, tab: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.tab_proj(tab))
        h = h + self.tab_res1(h)   # residual block 1
        h = h + self.tab_res2(h)   # residual block 2
        return h

    def fuse(self, z_tab, z_txt, gate_net, norm):
        """Task-conditioned gating: learns how much to trust tabular vs. text."""
        g = gate_net(torch.cat([z_tab, z_txt], dim=1))
        h = g * z_tab + (1 - g) * z_txt
        return norm(h), g

    def forward(self, input_ids, attention_mask, tab):
        out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)

        # Learned attention pooling (clinically-aware weighting of BERT tokens)
        cls = self.attn_pool(out.last_hidden_state, attention_mask)

        z_tab = self._encode_tab(tab)
        z_txt = self.text_proj(cls)

        # Shared transformation before task-specific gates
        z_tab = self.shared(z_tab)
        z_txt = self.shared(z_txt)

        h_type, g_type = self.fuse(z_tab, z_txt, self.gate_type, self.norm_type)
        h_risk, g_risk = self.fuse(z_tab, z_txt, self.gate_risk, self.norm_risk)
        h_side, g_side = self.fuse(z_tab, z_txt, self.gate_side, self.norm_side)
        h_sev,  g_sev  = self.fuse(z_tab, z_txt, self.gate_sev,  self.norm_sev)

        logits_type = self.head_type(h_type)
        logits_risk = self.head_risk(h_risk)
        logits_side = self.head_side(h_side)
        mu = self.head_sev_mu(h_sev).squeeze(1)
        lv = self.head_sev_lv(h_sev).squeeze(1)

        gates = {
            "type": g_type.mean().item(),
            "risk": g_risk.mean().item(),
            "side": g_side.mean().item(),
            "sev":  g_sev.mean().item(),
        }
        return logits_type, logits_risk, logits_side, mu, lv, gates


tab_dim = train_tab.shape[1]
model = GatedFusionMTL(tab_dim=tab_dim, text_model_name=MODEL_NAME, hidden=512, dropout=0.3).to(device)
print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Model parameters : 113,274,254
Trainable params : 113,274,254


## 8) Losses + Clinical Consistency Regularizer
**Soft rule:** high IgE should not confidently predict LOW side-effect risk.
This is a penalty, not a hard rule.

In [13]:

class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance in urticaria type & side-effect proxy.

    Down-weights easy examples so training focuses on hard / minority cases.
    gamma=2 is the standard value (Lin et al., ICCV 2017 – RetinaNet).
    Combined with class-weights this gives stronger imbalance correction than
    weighted CE + label smoothing alone.
    """
    def __init__(self, weight=None, gamma: float = 2.0, label_smoothing: float = 0.1):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
        self.ls     = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.ls, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# Class weights for urticaria type (imbalance handling)
type_counts = train_df[TARGET_TYPE].map(type2id).value_counts().sort_index().values
type_w = (type_counts.sum() / (type_counts + 1e-6))
type_w = torch.tensor(type_w / type_w.mean(), dtype=torch.float32).to(device)

# Side-effect proxy class weights
side_counts = train_df["sidefx_proxy"].value_counts().sort_index().values
side_w = (side_counts.sum() / (side_counts + 1e-6))
side_w = torch.tensor(side_w / side_w.mean(), dtype=torch.float32).to(device)

# Focal losses for both classification tasks
# Type: gamma=3.0 (higher than standard) to force harder focus on Mixed/Spontaneous;
#       label_smoothing=0.05 (lower) so the model is penalised more for wrong predictions.
# Side: gamma=2.0 (standard), label_smoothing=0.1 (unchanged — side-effect is already strong).
focal_type = FocalLoss(weight=type_w, gamma=3.0, label_smoothing=0.05)
focal_side = FocalLoss(weight=side_w, gamma=2.0, label_smoothing=0.1)

# Secondary risk: multi-label BCE (unchanged)
bce_risk = nn.BCEWithLogitsLoss()

def heteroscedastic_nll(y, mu, log_var):
    return 0.5 * (torch.exp(-log_var) * (y - mu)**2 + log_var)

# Clinical consistency penalty uses numeric IgE (scaled)
ig_num_idx = NUM_COLS.index("IgE")

# Derive threshold from training data (75th percentile of scaled IgE) instead of hardcoding
_ig_train_scaled = train_tab[:, ig_num_idx]
_ig_threshold = float(np.percentile(_ig_train_scaled, 75))
print(f"IgE penalty threshold (75th pct, scaled): {_ig_threshold:.4f}")

def clinical_consistency_penalty(tab_batch, side_probs, ig_threshold=_ig_threshold):
    ig = tab_batch[:, ig_num_idx]  # robust-scaled
    high_mask = (ig > ig_threshold).float()
    low_p = side_probs[:, 0]
    return (high_mask * low_p).mean()


IgE penalty threshold (75th pct, scaled): 0.5979


## 9) Optimizer (differential learning rates) + Training
This usually boosts type accuracy a lot: **small LR for BERT**, larger LR for fusion/heads.
We also use **early stopping** on validation Macro‑F1 for urticaria type.

In [14]:

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

EPOCHS             = 25
PATIENCE           = 6
FREEZE_BERT_EPOCHS = 2
ACCUM_STEPS        = 2
TAB_NOISE_STD      = 0.02

BERT_BASE_LR  = 2e-5
BERT_LR_DECAY = 0.9
HEAD_LR       = 1e-3
LAMBDA_RULE   = 0.15


def get_bert_layerwise_params(model, base_lr: float, decay: float):
    groups = []
    layers = model.text_encoder.encoder.layer
    n = len(layers)
    for i, layer in enumerate(layers):
        lr = base_lr * (decay ** (n - 1 - i))
        groups.append({"params": layer.parameters(), "lr": lr})
    groups.append({"params": model.text_encoder.embeddings.parameters(),
                   "lr": base_lr * (decay ** n)})
    if hasattr(model.text_encoder, "pooler"):
        groups.append({"params": model.text_encoder.pooler.parameters(), "lr": base_lr})
    return groups


bert_groups  = get_bert_layerwise_params(model, BERT_BASE_LR, BERT_LR_DECAY)
other_params = [p for n, p in model.named_parameters() if not n.startswith("text_encoder.")]

optimizer = torch.optim.AdamW(
    bert_groups + [{"params": other_params, "lr": HEAD_LR}],
    weight_decay=0.01,
)
scheduler  = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-7)
scaler_amp = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())


def evaluate(loader):
    model.eval()
    all_type_true, all_type_pred = [], []
    all_side_true, all_side_pred = [], []
    all_risk_true, all_risk_prob = [], []
    sev_true, sev_mu = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tab            = batch["tab"].to(device)
            y_type   = batch["y_type"].to(device)
            y_risk   = batch["y_risk"].to(device)
            y_side   = batch["y_side"].to(device)
            y_sev    = batch["sev"].to(device)
            sev_mask = batch["sev_mask"].to(device)

            logits_type, logits_risk, logits_side, mu, lv, gates = model(input_ids, attention_mask, tab)

            all_type_true.extend(y_type.cpu().numpy().tolist())
            all_type_pred.extend(torch.argmax(logits_type, dim=1).cpu().numpy().tolist())
            all_side_true.extend(y_side.cpu().numpy().tolist())
            all_side_pred.extend(torch.argmax(logits_side, dim=1).cpu().numpy().tolist())

            risk_prob = torch.sigmoid(logits_risk)
            all_risk_true.append(y_risk.cpu().numpy())
            all_risk_prob.append(risk_prob.cpu().numpy())

            m = sev_mask.bool()
            if m.any():
                sev_true.extend(y_sev[m].cpu().numpy().tolist())
                sev_mu.extend(mu[m].cpu().numpy().tolist())

    all_risk_true = np.concatenate(all_risk_true, axis=0)
    all_risk_prob = np.concatenate(all_risk_prob, axis=0)

    type_acc = accuracy_score(all_type_true, all_type_pred)
    type_f1  = f1_score(all_type_true, all_type_pred, average="macro")
    side_acc = accuracy_score(all_side_true, all_side_pred)
    side_f1  = f1_score(all_side_true, all_side_pred, average="macro")

    risk_aucs = []
    for i in range(all_risk_true.shape[1]):
        try:
            risk_aucs.append(roc_auc_score(all_risk_true[:, i], all_risk_prob[:, i]))
        except Exception:
            risk_aucs.append(np.nan)

    sev_true = np.array(sev_true)
    sev_mu   = np.array(sev_mu)
    mae = float(np.mean(np.abs(sev_true - sev_mu))) if len(sev_true) else None

    return {
        "type_acc":            float(type_acc),
        "type_f1_macro":       float(type_f1),
        "side_acc":            float(side_acc),
        "side_f1_macro":       float(side_f1),
        "risk_auc_thyroid":    None if np.isnan(risk_aucs[0]) else float(risk_aucs[0]),
        "risk_auc_autoimmune": None if np.isnan(risk_aucs[1]) else float(risk_aucs[1]),
        "severity_mae":        mae,
    }


def composite_score(m):
    """55% type F1 + 15% side F1 + 20% mean AUC + 10% severity.

    Severity term = max(0, 1 - MAE/5): MAE≤1→1.0, MAE=2.5→0.5, MAE≥5→0.0.
    Prevents checkpoint selection from picking models with degraded severity
    (MAE drifting up due to log_var instability in later epochs).
    """
    valid_aucs = [v for v in [m["risk_auc_thyroid"], m["risk_auc_autoimmune"]] if v is not None]
    mean_auc  = float(np.mean(valid_aucs)) if valid_aucs else 0.5
    sev_score = max(0.0, 1.0 - (m["severity_mae"] or 5.0) / 5.0)
    return (0.55 * m["type_f1_macro"]
          + 0.15 * m["side_f1_macro"]
          + 0.20 * mean_auc
          + 0.10 * sev_score)


history    = []  # record val metrics per epoch
best_score = -1
bad_epochs = 0
best_path  = "/content/best_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()

    if epoch <= FREEZE_BERT_EPOCHS:
        for p in model.text_encoder.parameters():
            p.requires_grad = False
    elif epoch == FREEZE_BERT_EPOCHS + 1:
        for p in model.text_encoder.parameters():
            p.requires_grad = True
        print(f"Epoch {epoch}: BERT unfrozen — full model training begins")

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        tab            = batch["tab"].to(device)
        y_type   = batch["y_type"].to(device)
        y_risk   = batch["y_risk"].to(device)
        y_side   = batch["y_side"].to(device)
        y_sev    = batch["sev"].to(device)
        sev_mask = batch["sev_mask"].to(device)

        if TAB_NOISE_STD > 0:
            tab = tab + torch.randn_like(tab) * TAB_NOISE_STD

        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", enabled=True):
            logits_type, logits_risk, logits_side, mu, lv, gates = model(input_ids, attention_mask, tab)

            L_type = focal_type(logits_type, y_type)
            L_risk = bce_risk(logits_risk, y_risk)
            L_side = focal_side(logits_side, y_side)

            nll   = heteroscedastic_nll(y_sev, mu, lv)
            L_sev = (nll * sev_mask).sum() / (sev_mask.sum() + 1e-6)

            side_probs = torch.softmax(logits_side, dim=1)
            L_rule     = clinical_consistency_penalty(tab, side_probs)

            # Clamp log_var to [-3, 3] — prevents uncertainty collapse that
            # causes negative total loss and severity MAE degradation late in training.
            lv_t = model.log_var.clamp(-3.0, 3.0)
            task_loss = (
                torch.exp(-lv_t[0]) * L_type + lv_t[0] +
                torch.exp(-lv_t[1]) * L_risk + lv_t[1] +
                torch.exp(-lv_t[2]) * L_side + lv_t[2] +
                torch.exp(-lv_t[3]) * L_sev  + lv_t[3]
            ) * 0.5

            loss = task_loss + LAMBDA_RULE * L_rule
            loss = loss / ACCUM_STEPS

        scaler_amp.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            scheduler.step(epoch - 1 + step / len(train_loader))
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix({
            "loss":      round(float(loss.item() * ACCUM_STEPS), 4),
            "gate_type": round(gates["type"], 3),
            "σ²_type":   round(float(torch.exp(lv_t[0]).item()), 3),
        })

    val_metrics = evaluate(val_loader)
    val_score   = composite_score(val_metrics)
    history.append({**val_metrics, "epoch": epoch, "composite": val_score})
    print(f"VAL [epoch={epoch}, composite={val_score:.4f}]:", val_metrics)

    if val_score > best_score:
        best_score = val_score
        bad_epochs = 0
        torch.save(model.state_dict(), best_path)
        print("  ✅ Saved best model:", best_path)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("  ⏹  Early stopping triggered.")
            break

print(f"\nBest composite val score: {best_score:.4f}")

# ── Post-training: per-class probability calibration ─────────────────────────
# WeightedRandomSampler balances the batches, but the model's raw softmax
# probabilities may still be biased. Grid-search per-class scale factors on
# the validation set that maximise macro-F1; applied at test time only.
# Scales are multiplicative: pred = argmax( softmax(logits) ⊙ scale )
print("\nCalibrating type-class probability scales on validation set...")
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

_val_probs, _val_true = [], []
with torch.no_grad():
    for batch in val_loader:
        logits_type, _, _, _, _, _ = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["tab"].to(device))
        _val_probs.append(torch.softmax(logits_type, dim=1).cpu().numpy())
        _val_true.extend(batch["y_type"].numpy())

_val_probs = np.concatenate(_val_probs)
_val_true  = np.array(_val_true)

_base_f1 = f1_score(_val_true, np.argmax(_val_probs, axis=1), average="macro")
best_cal_f1, type_scales = _base_f1, np.ones(len(type_classes))

for s0 in np.arange(0.5, 2.1, 0.1):
    for s1 in np.arange(0.5, 2.1, 0.1):
        for s2 in np.arange(0.5, 2.1, 0.1):
            scale = np.array([s0, s1, s2])
            preds = np.argmax(_val_probs * scale, axis=1)
            f1    = f1_score(_val_true, preds, average="macro", zero_division=0)
            if f1 > best_cal_f1:
                best_cal_f1  = f1
                type_scales  = scale.copy()

print(f"  Argmax (no cal) val macro-F1 : {_base_f1:.4f}")
print(f"  Calibrated      val macro-F1 : {best_cal_f1:.4f}")
print(f"  Optimal scales  [{id2type[0]}, {id2type[1]}, {id2type[2]}]: {type_scales.round(2)}")


Epoch 1/25:   0%|          | 0/129 [00:00<?, ?it/s]

VAL [epoch=1, composite=0.6342]: {'type_acc': 0.5929752066115702, 'type_f1_macro': 0.5551782682512734, 'side_acc': 0.6838842975206612, 'side_f1_macro': 0.633222736536, 'risk_auc_thyroid': 0.8241598923005229, 'risk_auc_autoimmune': 0.9872385086945114, 'severity_mae': 2.365326629197302}
  ✅ Saved best model: /content/best_model.pt


Epoch 2/25:   0%|          | 0/129 [00:00<?, ?it/s]

VAL [epoch=2, composite=0.7075]: {'type_acc': 0.6136363636363636, 'type_f1_macro': 0.6525075712978659, 'side_acc': 0.7665289256198347, 'side_f1_macro': 0.7168168168168169, 'risk_auc_thyroid': 0.9038816686515128, 'risk_auc_autoimmune': 0.9987834840063927, 'severity_mae': 2.4562284739549494}
  ✅ Saved best model: /content/best_model.pt
Epoch 3: BERT unfrozen — full model training begins


Epoch 3/25:   0%|          | 0/129 [00:00<?, ?it/s]

...

VAL [epoch=9, composite=0.7144]: {'type_acc': 0.6074380165289256, 'type_f1_macro': 0.6618757811226431, 'side_acc': 0.8119834710743802, 'side_f1_macro': 0.7714054129338996, 'risk_auc_thyroid': 0.8990144807469924, 'risk_auc_autoimmune': 0.9987357774968395, 'severity_mae': 2.7537023528548312}


Epoch 10/25:   0%|          | 0/129 [00:00<?, ?it/s]

VAL [epoch=10, composite=0.7235]: {'type_acc': 0.5991735537190083, 'type_f1_macro': 0.656357049660154, 'side_acc': 0.8863636363636364, 'side_f1_macro': 0.8598381896118297, 'risk_auc_thyroid': 0.9026735014411708, 'risk_auc_autoimmune': 0.9989266035350524, 'severity_mae': 2.8319588576959185}


Epoch 11/25:   0%|          | 0/129 [00:00<?, ?it/s]

VAL [epoch=11, composite=0.7337]: {'type_acc': 0.6053719008264463, 'type_f1_macro': 0.6756942419234632, 'side_acc': 0.890495867768595, 'side_f1_macro': 0.8584232683769967, 'risk_auc_thyroid': 0.8983068399523636, 'risk_auc_autoimmune': 0.9983064189108604, 'severity_mae': 2.818381540662001}
  ⏹  Early stopping triggered.

Best composite val score: 0.7464

Calibrating type-class probability scales on validation set...
  Argmax (no cal) val macro-F1 : 0.6906
  Calibrated      val macro-F1 : 0.6906
  Optimal scales  [Inducible, Mixed, Spontaneous]: [1. 1. 1.]


## 10) Test evaluation + reports

In [15]:

model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate(test_loader)
print("TEST:", test_metrics)

# Detailed reports for main tasks
def get_preds(loader, type_scales=None):
    """
    Get predictions for all tasks.

    Parameters
    ----------
    type_scales : array-like of shape (n_type_classes,), optional
        Per-class probability scale factors found by post-training calibration.
        pred = argmax( softmax(logits_type) ⊙ type_scales )
        When None, falls back to raw argmax (equivalent to type_scales=all-ones).
    """
    model.eval()
    all_type_true, all_type_pred = [], []
    all_side_true, all_side_pred = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tab = batch["tab"].to(device)

            y_type = batch["y_type"].to(device)
            y_side = batch["y_side"].to(device)

            logits_type, logits_risk, logits_side, mu, lv, gates = model(input_ids, attention_mask, tab)

            all_type_true.extend(y_type.cpu().numpy().tolist())

            # Apply per-class calibration scales before taking argmax.
            # This corrects residual probability bias from class imbalance
            # without retraining. Uses scales found on validation set.
            probs_type = torch.softmax(logits_type, dim=1).cpu()
            if type_scales is not None:
                probs_type = probs_type * torch.tensor(type_scales, dtype=torch.float32)
            all_type_pred.extend(probs_type.argmax(dim=1).numpy().tolist())

            all_side_true.extend(y_side.cpu().numpy().tolist())
            all_side_pred.extend(torch.argmax(logits_side, dim=1).cpu().numpy().tolist())

    return all_type_true, all_type_pred, all_side_true, all_side_pred

# Use calibrated type_scales from post-training threshold search
tt, tp, st, sp = get_preds(test_loader, type_scales=type_scales)

print("\n=== Urticaria Type Report (calibrated) ===")
print("Accuracy:", accuracy_score(tt, tp))
print(classification_report(tt, tp, target_names=[id2type[i] for i in range(len(type_classes))]))

print("\n=== Side-effect Proxy Report (LOW/MODERATE/HIGH) ===")
print("Accuracy:", accuracy_score(st, sp))
print(classification_report(st, sp, target_names=["LOW","MODERATE","HIGH"]))


# ══════════════════════════════════════════════════════════════════════════════
# 10b) CLINICAL RISK PROFILE GENERATOR
# Combines all 4 MTL task outputs into a structured, interpretable patient-
# level risk summary with composite score and clinical interpretation text.
# ══════════════════════════════════════════════════════════════════════════════

_SIDE_LABELS = ["LOW", "MODERATE", "HIGH"]
_SEV_BANDS   = [
    (0.0,  3.0, "MILD",     "Minimal impact; standard antihistamine management"),
    (3.0,  6.0, "MODERATE", "Meaningful impairment; scheduled follow-up recommended"),
    (6.0,  8.0, "SEVERE",   "Significant burden; consider second-line / biologic therapy"),
    (8.0, 10.0, "EXTREME",  "Debilitating; urgent escalation to specialist required"),
]

def _sev_band(score: float):
    for lo, hi, label, desc in _SEV_BANDS:
        if lo <= score < hi:
            return label, desc
    return _SEV_BANDS[-1][2], _SEV_BANDS[-1][3]

def _composite_interpretation(score: float) -> str:
    if score >= 0.75: return "VERY HIGH — Urgent specialist referral; escalate therapy immediately."
    if score >= 0.55: return "HIGH      — Close monitoring; consider biologics (e.g. omalizumab)."
    if score >= 0.35: return "MODERATE  — Regular review every 4 weeks; optimise current therapy."
    return              "LOW       — Standard management; annual review sufficient."

def make_risk_profile(model, sample_batch: dict, type_classes: list,
                      type_scales=None) -> dict:
    """
    Generate a structured clinical risk profile for a single patient.
    All 4 task outputs are combined into one interpretable report.

    Parameters
    ----------
    model        : trained GatedFusionMTL (set to eval externally)
    sample_batch : dict of tensors on device, each with batch dim = 1
    type_classes : list[str] – urticaria type label strings
    type_scales  : optional calibration scales for type probabilities
    """
    model.eval()
    with torch.no_grad():
        logits_type, logits_risk, logits_side, mu, lv, gates = model(
            sample_batch["input_ids"],
            sample_batch["attention_mask"],
            sample_batch["tab"],
        )

    type_probs = torch.softmax(logits_type, dim=-1).cpu().squeeze()
    if type_scales is not None:
        cal_probs    = type_probs * torch.tensor(type_scales, dtype=torch.float32)
        type_pred_id = int(cal_probs.argmax())
    else:
        type_pred_id = int(type_probs.argmax())

    risk_probs   = torch.sigmoid(logits_risk).cpu().squeeze()
    side_probs   = torch.softmax(logits_side, dim=-1).cpu().squeeze()
    side_pred_id = int(side_probs.argmax())

    sev_mean  = float(mu.cpu().item())
    sev_std   = float(torch.sqrt(torch.exp(lv.cpu())).item())
    sev_clamp = max(0.0, min(10.0, sev_mean))
    sev_label, sev_desc = _sev_band(sev_clamp)

    # Composite risk score (0–1):
    #   35% HIGH side-effect probability
    #   25% maximum secondary disease risk probability
    #   25% normalised severity (0–10 → 0–1)
    #   15% urticaria type prediction confidence
    composite = (
        0.35 * float(side_probs[2]) +
        0.25 * float(risk_probs.max()) +
        0.25 * (sev_clamp / 10.0) +
        0.15 * float(type_probs.max())
    )

    return {
        "urticaria_type": {
            "predicted":    type_classes[type_pred_id],
            "confidence_pct": round(float(type_probs[type_pred_id]) * 100, 1),
            "distribution": {type_classes[i]: round(float(p)*100,1) for i,p in enumerate(type_probs)},
        },
        "secondary_disease_risk": {
            "thyroid_risk_pct":    round(float(risk_probs[0]) * 100, 1),
            "autoimmune_risk_pct": round(float(risk_probs[1]) * 100, 1),
            "thyroid_flag":        bool(risk_probs[0] > 0.50),
            "autoimmune_flag":     bool(risk_probs[1] > 0.50),
        },
        "sideeffect_risk": {
            "level":          _SIDE_LABELS[side_pred_id],
            "distribution":   {_SIDE_LABELS[i]: round(float(p)*100,1) for i,p in enumerate(side_probs)},
            "high_risk_flag": bool(side_probs[2] > 0.40),
        },
        "severity": {
            "predicted_score": round(sev_clamp, 2),
            "uncertainty_95ci": (round(max(0.0, sev_mean - 1.96*sev_std), 2),
                                 round(min(10.0, sev_mean + 1.96*sev_std), 2)),
            "band":        sev_label,
            "description": sev_desc,
        },
        "composite_risk_score":    round(composite, 3),
        "clinical_interpretation": _composite_interpretation(composite),
        "modality_gates":          {k: round(v, 3) for k, v in gates.items()},
    }


def print_risk_profile(profile: dict):
    """Pretty-print a structured clinical risk profile as a clinical report."""
    print(f"\n{'═'*64}")
    print("   CHRONIC URTICARIA  |  CLINICAL RISK PROFILE")
    print(f"{'═'*64}")

    u = profile["urticaria_type"]
    print(f"\n[1] URTICARIA TYPE")
    print(f"    Predicted   : {u['predicted']}  ({u['confidence_pct']}% confidence)")
    dist = " | ".join(f"{k}: {v}%" for k,v in u["distribution"].items())
    print(f"    Distribution: {dist}")

    r = profile["secondary_disease_risk"]
    print(f"\n[2] SECONDARY DISEASE RISK")
    t_flag = "⚠  ELEVATED" if r["thyroid_flag"]    else "✓  Normal"
    a_flag = "⚠  ELEVATED" if r["autoimmune_flag"] else "✓  Normal"
    print(f"    Thyroid risk     : {r['thyroid_risk_pct']:5.1f}%   [{t_flag}]")
    print(f"    Autoimmune risk  : {r['autoimmune_risk_pct']:5.1f}%   [{a_flag}]")

    s = profile["sideeffect_risk"]
    hi = "  ⚠" if s["high_risk_flag"] else ""
    print(f"\n[3] SIDE-EFFECT / HYPERSENSITIVITY RISK{hi}")
    print(f"    Level       : {s['level']}")
    dist = " | ".join(f"{k}: {v}%" for k,v in s["distribution"].items())
    print(f"    Distribution: {dist}")

    sv = profile["severity"]
    ci_lo, ci_hi = sv['uncertainty_95ci']
    print(f"\n[4] SYMPTOM SEVERITY  (itching score / 10)")
    print(f"    Predicted   : {sv['predicted_score']} / 10   [{sv['band']}]")
    print(f"    Range (95%) : {ci_lo} – {ci_hi} / 10")
    print(f"    Description : {sv['description']}")

    cs = profile["composite_risk_score"]
    bar_len = int(cs * 40)
    bar = "█" * bar_len + "░" * (40 - bar_len)
    print(f"\n[5] COMPOSITE RISK SCORE")
    print(f"    [{bar}]  {cs:.3f}")

    print(f"\n[6] CLINICAL INTERPRETATION")
    print(f"    {profile['clinical_interpretation']}")

    print(f"\n[7] MODALITY GATE WEIGHTS  (tab vs. text blend per task)")
    for task, gate in profile["modality_gates"].items():
        t_pct = int(gate * 100)
        x_pct = 100 - t_pct
        bt = "▓" * (t_pct // 5)
        bx = "░" * (x_pct // 5)
        print(f"    {task:<8}: Tab [{bt:<20}] {t_pct:3d}%  Text [{bx:<20}] {x_pct:3d}%")
    print(f"\n{'─'*64}\n")


# ── Demo: risk profile for first test sample ──────────────────────────────────
sample       = test_ds[0]
sample_batch = {k: v.unsqueeze(0).to(device) for k, v in sample.items()}
profile      = make_risk_profile(model, sample_batch, type_classes, type_scales=type_scales)
print_risk_profile(profile)


TEST: {'type_acc': 0.6365638766519823, 'type_f1_macro': 0.6989964121187421, 'side_acc': 0.8964757709251101, 'side_f1_macro': 0.8839149763618455, 'risk_auc_thyroid': 0.9229724409448818, 'risk_auc_autoimmune': 0.9956123665322478, 'severity_mae': 2.7171520509383753}

=== Urticaria Type Report (calibrated) ===
Accuracy: 0.6365638766519823
              precision    recall  f1-score   support

   Inducible       1.00      1.00      1.00        78
       Mixed       0.58      0.68      0.62       201
 Spontaneous       0.54      0.42      0.47       175

    accuracy                           0.64       454
   macro avg       0.70      0.70      0.70       454
weighted avg       0.63      0.64      0.63       454


=== Side-effect Proxy Report (LOW/MODERATE/HIGH) ===
Accuracy: 0.8964757709251101
              precision    recall  f1-score   support

         LOW       0.91      0.95      0.93       192
    MODERATE       0.94      0.83      0.88       207
        HIGH       0.75      0.96   

## 11) Training History & Results Visualisation

Plots are saved as PNG files alongside the notebook.
- `fusion_training_curves.png` — per-epoch validation metrics
- `fusion_test_metrics_bar.png` — bar chart of all test metrics
- `fusion_confusion_matrices.png` — type & side-effect confusion matrices

In [16]:
import matplotlib
matplotlib.use("Agg")          # headless — works in both local & Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Unpack history ────────────────────────────────────────────────────────────
epochs_x     = [h["epoch"]              for h in history]
type_acc_h   = [h["type_acc"]           for h in history]
type_f1_h    = [h["type_f1_macro"]      for h in history]
side_acc_h   = [h["side_acc"]           for h in history]
side_f1_h    = [h["side_f1_macro"]      for h in history]
composite_h  = [h["composite"]          for h in history]
auc_thy_h    = [h["risk_auc_thyroid"]   if h["risk_auc_thyroid"]   is not None else float("nan") for h in history]
auc_auto_h   = [h["risk_auc_autoimmune"] if h["risk_auc_autoimmune"] is not None else float("nan") for h in history]
sev_mae_h    = [h["severity_mae"]       if h["severity_mae"]       is not None else float("nan") for h in history]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Gated Fusion MTL — Validation Metrics per Epoch", fontsize=14, fontweight="bold")

_style = dict(linewidth=2, marker="o", markersize=4)

# Row 0: Type
axes[0,0].plot(epochs_x, type_acc_h, color="#2196F3", label="Type Accuracy", **_style)
axes[0,0].plot(epochs_x, type_f1_h,  color="#FF9800", linestyle="--", label="Type F1 (macro)", **_style)
axes[0,0].set_title("Urticaria Type")
axes[0,0].set_ylabel("Score")
axes[0,0].legend(fontsize=8)
axes[0,0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Row 0: Side-effect
axes[0,1].plot(epochs_x, side_acc_h, color="#4CAF50", label="Side-eff Accuracy", **_style)
axes[0,1].plot(epochs_x, side_f1_h,  color="#F44336", linestyle="--", label="Side-eff F1 (macro)", **_style)
axes[0,1].set_title("Side-Effect Proxy")
axes[0,1].set_ylabel("Score")
axes[0,1].legend(fontsize=8)
axes[0,1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Row 1: Risk AUCs
axes[1,0].plot(epochs_x, auc_thy_h,  color="#9C27B0", label="Thyroid AUC", **_style)
axes[1,0].plot(epochs_x, auc_auto_h, color="#00BCD4", linestyle="--", label="Autoimmune AUC", **_style)
axes[1,0].axhline(0.5, color="gray", linestyle=":", linewidth=1, label="Random")
axes[1,0].set_title("Secondary Disease Risk AUC")
axes[1,0].set_ylabel("AUC-ROC")
axes[1,0].legend(fontsize=8)
axes[1,0].set_ylim(0.4, 1.05)

# Row 1: Severity MAE
axes[1,1].plot(epochs_x, sev_mae_h, color="#795548", **_style)
axes[1,1].set_title("Severity MAE (lower is better)")
axes[1,1].set_ylabel("MAE (0–10 scale)")
axes[1,1].invert_yaxis()

# Row 2: Composite score (spans full width)
ax_comp = axes[2,0]
ax_comp.plot(epochs_x, composite_h, color="#E91E63", linewidth=2.5, marker="o", markersize=5)
best_idx = int(np.nanargmax(composite_h))
ax_comp.axvline(epochs_x[best_idx], color="gray", linestyle="--", linewidth=1)
ax_comp.annotate(f"Best\n{composite_h[best_idx]:.4f}",
                 xy=(epochs_x[best_idx], composite_h[best_idx]),
                 xytext=(epochs_x[best_idx]+0.5, composite_h[best_idx]-0.02),
                 fontsize=8, color="#E91E63")
ax_comp.set_title("Composite Score (0.55×TypeF1 + 0.15×SideF1 + 0.20×AUC + 0.10×SevScore)")
ax_comp.set_ylabel("Composite")
ax_comp.set_xlabel("Epoch")

axes[2,1].axis("off")  # empty

for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=8)

plt.tight_layout()
save_path = "fusion_training_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")


Saved → fusion_training_curves.png


### Test-Set Performance Plots

In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from sklearn.metrics import confusion_matrix

# ── 1) Bar chart of all test metrics ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Gated Fusion MTL — Test-Set Metrics", fontsize=13, fontweight="bold")

# Classification + AUC metrics
clf_names  = ["Type\nAccuracy", "Type\nF1 (macro)", "Side\nAccuracy", "Side\nF1 (macro)", "Thyroid\nAUC", "Autoimmune\nAUC"]
clf_values = [
    test_metrics["type_acc"],
    test_metrics["type_f1_macro"],
    test_metrics["side_acc"],
    test_metrics["side_f1_macro"],
    test_metrics["risk_auc_thyroid"]   or 0.0,
    test_metrics["risk_auc_autoimmune"] or 0.0,
]
colors = ["#2196F3","#FF9800","#4CAF50","#F44336","#9C27B0","#00BCD4"]

bars = axes[0].bar(clf_names, clf_values, color=colors, edgecolor="white", linewidth=0.8, width=0.6)
axes[0].set_ylim(0, 1.12)
axes[0].axhline(1.0, color="gray", linestyle=":", linewidth=0.8)
for bar, val in zip(bars, clf_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.015,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_title("Classification & AUC Metrics")
axes[0].set_ylabel("Score")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].tick_params(axis="x", labelsize=8)
axes[0].grid(axis="y", alpha=0.3)

# Severity MAE (separate axis — different scale)
mae_val = test_metrics["severity_mae"] or 0.0
bar2 = axes[1].bar(["Severity\nMAE (↓)"], [mae_val], color="#795548", edgecolor="white", width=0.4)
axes[1].set_ylim(0, max(5.0, mae_val * 1.4))
axes[1].text(bar2[0].get_x() + bar2[0].get_width()/2, mae_val + 0.05,
             f"{mae_val:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
axes[1].set_title("Severity Regression\n(MAE on 0–10 scale)")
axes[1].set_ylabel("MAE")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
bar_path = "fusion_test_metrics_bar.png"
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {bar_path}")

# ── 2) Confusion matrices ──────────────────────────────────────────────────────
type_labels = [id2type[i] for i in range(len(type_classes))]
side_labels = ["LOW", "MODERATE", "HIGH"]

cm_type = confusion_matrix(tt, tp)
cm_side = confusion_matrix(st, sp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Gated Fusion MTL — Test Confusion Matrices", fontsize=13, fontweight="bold")

def plot_cm(ax, cm, labels, title):
    total = cm.sum()
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Predicted", fontsize=10); ax.set_ylabel("True", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            pct = cm[i, j] / total * 100
            ax.text(j, i, f"{cm[i,j]}\n({pct:.1f}%)",
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=8)

plot_cm(axes[0], cm_type, type_labels, "Urticaria Type Classification")
plot_cm(axes[1], cm_side, side_labels, "Side-Effect Risk Level")

plt.tight_layout()
cm_path = "fusion_confusion_matrices.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {cm_path}")


Saved → fusion_test_metrics_bar.png
Saved → fusion_confusion_matrices.png


## 11) Save artifacts for backend (model + preprocess + config)
You will load these in FastAPI later.

In [18]:

ART_DIR = "/content/artifacts"
os.makedirs(ART_DIR, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), f"{ART_DIR}/model.pt")

# Save preprocess pipeline (scaler + onehot)
joblib.dump(preprocess, f"{ART_DIR}/preprocess.joblib")

# Save config
config = {
    "text_model": MODEL_NAME,
    "max_len": MAX_LEN,
    "type_classes": type_classes,
    "risk_labels": RISK_LABELS,
    "sidefx_classes": ["LOW", "MODERATE", "HIGH"],
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
    "excluded_leak_cols": LEAK_COLS,
    "note_text_cols": TEXT_COLS,
    "model_arch": {
        "hidden": 512,
        "dropout": 0.3,
        "attention_pooling": True,
        "tab_residual_blocks": 2,
        "mlp_task_heads": True,
        "uncertainty_mtl": True,
    },
}
with open(f"{ART_DIR}/config.json","w") as f:
    json.dump(config, f, indent=2)

print("✅ Artifacts saved to:", ART_DIR)


✅ Artifacts saved to: /content/artifacts


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:

# Zip artifacts for download
import shutil
from google.colab import files

shutil.make_archive("/content/cu_artifacts", "zip", ART_DIR)
files.download("/content/cu_artifacts.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>